# faro on the Ti2-E: a first timelapse

A minimal real-time experiment on the Ti2-E: one field of view at the
current stage position, imaged every few seconds. Every frame is
segmented, tracked and measured while the next one is acquired. There
is no stimulation.

Before you start, put a sample with a nuclear marker (for example
H2B-miRFP) in focus and leave the stage at the field you want.

## 1. Connect to the Ti2-E

`Ti2E()` loads `NikonTi2e.cfg`, applies the `System/Startup` preset and
selects the `Channel` group. Pass `config=r"..."` to load another file.
Close NIS-Elements and any other Micro-Manager first: a device can only be
opened by one program at a time.

In [ ]:
from faro_aoki.ti2e import Ti2E
import faro.core.utils as utils

mic = Ti2E()
utils.print_configs(mic.mmc)

## 2. Settings

`CHANNEL` must be a preset of the `Channel` group listed above.
`WRITER` picks the storage format: `"omezarr"` writes one OME-Zarr
store, `"tiff"` one TIFF file per frame. Every run gets its own
folder, so nothing is overwritten.

In [ ]:
import os
import tempfile
from datetime import datetime

CHANNEL = {"config": "iRFP", "exposure": 100}  # nuclear marker
INTERVAL_S = 5.0  # seconds between frames
N_FRAMES = 10
WRITER = "omezarr"  # or "tiff"

path = os.path.join(
    tempfile.gettempdir(), f"faro-ti2e-demo-{datetime.now():%Y%m%d-%H%M%S}"
)

## 3. Set up the processing pipeline

Otsu thresholding for segmentation, trackpy for tracking and
`SimpleFE` for area and intensity per cell.

In [ ]:
from faro.core.data_structures import SegmentationMethod
from faro.core.pipeline import ImageProcessingPipeline
from faro.feature_extraction.simple import SimpleFE
from faro.segmentation.base import OtsuSegmentator
from faro.tracking.trackpy import TrackerTrackpy

segmentators = [
    SegmentationMethod(
        name="labels",
        segmentation_class=OtsuSegmentator(),
        use_channel=0,
        save_tracked=True,
    )
]

pipeline = ImageProcessingPipeline(
    storage_path=path,
    segmentators=segmentators,
    feature_extractor=SimpleFE("labels"),
    tracker=TrackerTrackpy(),
)

## 4. Preview segmentation

Snap one frame at the current position and segment it. Adjust the
exposure or focus until the labels look right.

In [ ]:
import matplotlib.pyplot as plt

mic.mmc.setConfig(mic.CHANNEL_GROUP, CHANNEL["config"])
mic.mmc.setExposure(CHANNEL["exposure"])
mic.mmc.snapImage()
test_img = mic.mmc.getImage()
labels = segmentators[0].segmentation_class.segment(test_img)

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].imshow(test_img, cmap="gray")
axes[0].set_title(CHANNEL["config"])
axes[1].imshow(labels, cmap="nipy_spectral")
axes[1].set_title(f"Labels ({labels.max()} objects)")
for ax in axes:
    ax.axis("off")
plt.tight_layout()

## 5. Define the experiment

The position is the current stage position. It has no `z`, so faro
never moves the focus drive: the focus stays where it is, or where
the PFS holds it.

In [ ]:
from faro.core.data_structures import RTMSequence

x0, y0 = mic.mmc.getXYPosition()
seq = RTMSequence(
    time_plan={"interval": INTERVAL_S, "loops": N_FRAMES},
    stage_positions=[{"x": x0, "y": y0}],
    channels=[CHANNEL],
)

events = list(seq)
print(f"{len(events)} events at x={x0:.1f} um, y={y0:.1f} um")
events[0]

## 6. Run the experiment

`validate_events` checks the channel names and exposures against the
hardware before anything moves.

In [ ]:
from faro.core.controller import Controller
from faro.core.writers import OmeZarrWriter, TiffWriter

if WRITER == "omezarr":
    writer = OmeZarrWriter(storage_path=path)
else:
    writer = TiffWriter(storage_path=path)

ctrl = Controller(mic, pipeline, writer=writer)
assert ctrl.validate_events(events), "fix the warnings above before running"
ctrl.run_experiment(events).wait()
ctrl.finish_experiment()
print("results in", path)

## 7. Inspect the results

Tracks are one parquet file per field of view. The images are in
`acquisition.ome.zarr` (OME-Zarr) or in `raw/` and `labels/`, one
file per frame (TIFF).

In [ ]:
import pandas as pd

tracks = pd.read_parquet(os.path.join(path, "tracks", "0_latest.parquet"))
print(
    f"Tracked {tracks['particle'].nunique()} cells across "
    f"{tracks['timestep'].nunique()} frames"
)
tracks.head()

In [ ]:
from glob import glob

import numpy as np
import tifffile
import zarr


def first_channel(frame):
    """First channel of a (y, x) or (c, y, x) frame."""
    frame = np.asarray(frame)
    return frame if frame.ndim == 2 else frame[0]


if WRITER == "omezarr":
    store = zarr.open(os.path.join(path, "acquisition.ome.zarr"), mode="r")
    raw = store["0"]  # (t, c, y, x) for one position
    lbl = store["labels/labels/0"]  # (t, y, x)
    first, last, first_lbl = raw[0], raw[-1], lbl[0]
else:
    raw_files = sorted(glob(os.path.join(path, "raw", "*.tiff")))
    lbl_files = sorted(glob(os.path.join(path, "labels", "*.tiff")))
    first, last = tifffile.imread(raw_files[0]), tifffile.imread(raw_files[-1])
    first_lbl = tifffile.imread(lbl_files[0])

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
axes[0].imshow(first_channel(first), cmap="gray")
axes[0].set_title("Raw (first frame)")
axes[1].imshow(first_channel(last), cmap="gray")
axes[1].set_title("Raw (last frame)")
axes[2].imshow(first_lbl, cmap="nipy_spectral")
axes[2].set_title("Labels (first frame)")
for ax in axes:
    ax.axis("off")
plt.tight_layout()